# Guía del módulo `acquisition.py` · descarga automática de archivos

Este notebook documenta y prueba el módulo de Python [`acquisition.py`](./acquisition.py), que está en esta misma carpeta (`src/shared/`). Es el código que usan los seis notebooks de descarga de Bronze para traer los archivos de las fuentes oficiales.

**Qué hace el módulo, en tres ideas**

1. **Descarga con reintentos.** Hasta 3 intentos si la fuente falla por red o por un error del servidor.
2. **Archivo histórico que solo crece (append-only).** Cada archivo se guarda tal como llegó, en una carpeta por año y mes. Nunca se sobrescribe ni se borra nada.
3. **Idempotencia con SHA-256.** Si el mismo contenido ya está guardado, no se vuelve a escribir. El Job puede correr dos veces sin crear duplicados.

**Cómo usar este notebook:** conéctalo a **Serverless** y presiona **Run all**. Las pruebas escriben solo en una carpeta temporal del clúster: **no tocan el Volume real ni las tablas de la plataforma**. La única celda que lee datos de la plataforma es la consulta de la sección 8.

| Sección | Función | Para qué sirve |
|---|---|---|
| 1 | Importar el módulo | Cómo lo cargan los notebooks de descarga |
| 2 | `SourceUnavailable` | Separar "la fuente está caída" de "hay que arreglar el proyecto" |
| 3 | `http_get` | Descargar con reintentos |
| 4 | `ckan_resources` | Pedir la URL actual al portal datos.gob.do |
| 5 | `_sha256_file` | Huella del contenido de un archivo |
| 6 | `_archived_files` | Recorrer el archivo histórico |
| 7 | `land` | Guardar solo lo nuevo |
| 8 | `source_unavailable` | Registrar la caída y terminar como advertencia |
| 9 | Cómo lo usa un notebook de descarga | El patrón completo `try / except` |

## 1 · Importar el módulo

`acquisition.py` no es un notebook: es un **archivo de Python** (un módulo). No se ejecuta solo; los notebooks lo importan. Para poder hacer `import acquisition`, primero se agrega la carpeta `shared` a `sys.path`, que es la lista de carpetas donde Python busca módulos.

Los notebooks de Bronze están en `src/bronze/`, así que suben dos niveles (`rsplit("/", 2)`) y agregan `/shared`. Este notebook ya está dentro de `shared`, así que solo quita su propio nombre (`rsplit("/", 1)`).

`importlib.reload` obliga a leer de nuevo el archivo. Sin eso, si cambias `acquisition.py`, el notebook seguiría usando la versión que quedó en memoria.

In [0]:
import sys, importlib, inspect, hashlib, os, time, tempfile, urllib.error
from datetime import datetime
from zoneinfo import ZoneInfo

nb = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
carpeta_shared = "/Workspace" + nb.rsplit("/", 1)[0]
sys.path.insert(0, carpeta_shared)

import acquisition as ACQ
importlib.reload(ACQ)

print("Módulo cargado desde:", ACQ.__file__)
print("Funciones públicas:", [n for n, f in inspect.getmembers(ACQ, inspect.isfunction) if f.__module__ == "acquisition" and not n.startswith("_")])
print("Funciones internas:", [n for n, f in inspect.getmembers(ACQ, inspect.isfunction) if f.__module__ == "acquisition" and n.startswith("_")])

**Función para ver el código real.** En lugar de copiar el código aquí (y que se desactualice), las celdas usan `ver()`, que imprime la función directamente desde `acquisition.py` con `inspect.getsource`. Lo que lees siempre es lo que corre en producción.

In [0]:
def ver(objeto):
    """Imprime el código fuente actual de una función o clase del módulo."""
    print(inspect.getsource(objeto))

print(ACQ.__doc__)

## 2 · `SourceUnavailable`: una excepción propia

Una clase que hereda de `Exception` y no agrega nada más. Su valor está en el **nombre**: permite separar dos tipos de error.

| Tipo de error | Ejemplos | Qué pasa en el Job |
|---|---|---|
| **La fuente está caída** (no es culpa del proyecto) | Sin conexión, tiempo agotado, certificado TLS inválido, HTTP 5xx | `SourceUnavailable` → la tarea termina como **advertencia** y la fuente conserva sus últimos datos |
| **Hay que arreglar el proyecto** | HTTP 404 (archivo movido), 403, el Excel cambió de formato | El error sube → la tarea **falla en rojo** |

Los notebooks capturan **solo** `SourceUnavailable` con `except ACQ.SourceUnavailable`. Cualquier otro error sigue subiendo, que es lo que se quiere.

In [0]:
ver(ACQ.SourceUnavailable)
print("¿Es un tipo de Exception?", issubclass(ACQ.SourceUnavailable, Exception))

## 3 · `http_get`: descargar con reintentos

Lee el código y luego la explicación línea por línea.

In [0]:
ver(ACQ.http_get)

**Línea por línea**

1. `urllib.parse.quote(url, safe=":/?=&%#")`: convierte espacios y acentos del nombre del archivo a formato URL (`%20`, etc.). ProConsumidor publica archivos como `precios de productos de primera necesidad.csv`. `safe` deja intactos los caracteres que forman la URL.
2. `for attempt in range(1, attempts + 1)`: hasta 3 intentos (valor por defecto).
3. `with urllib.request.urlopen(...) as r: return r.read()`: el `with` cierra la conexión pase lo que pase. Si la descarga funciona, `return` sale de la función en ese mismo intento.
4. `except urllib.error.HTTPError` con `if e.code < 500: raise`: **la decisión más importante**.
   - **4xx** (404, 403): reintentar no sirve. `raise` relanza el error y la tarea falla en rojo.
   - **5xx** (500, 502, 503): problema temporal del servidor. Se guarda el error y se reintenta.
5. `except (URLError, TimeoutError, ConnectionError, OSError)`: errores de red, tiempo agotado y certificado TLS inválido. También se reintentan.
6. **El orden de los `except` importa.** `HTTPError` es una subclase de `URLError`. Python revisa los `except` de arriba hacia abajo; si `URLError` estuviera primero, atraparía también los 404 y los reintentaría sin sentido.
7. `time.sleep(10 * attempt)`: espera 10 s después del primer fallo y 20 s después del segundo (espera lineal). Después del último no espera.
8. `raise SourceUnavailable(...)`: si se agotan los intentos. `{last!r}` muestra el último error con su tipo.

**Lo que nunca se hace:** desactivar la verificación del certificado TLS. Un certificado vencido se trata como fuente no disponible.

In [0]:
# Comprobación del punto 6: HTTPError es un tipo de URLError
print("HTTPError es subclase de URLError:", issubclass(urllib.error.HTTPError, urllib.error.URLError))

### Prueba A · descarga real que funciona

Descarga el archivo del IPC del Banco Central (el mismo que usa `02_BCRD_CPI_Download`). Solo se guarda en memoria; no se escribe en ningún lado.

In [0]:
URL_IPC = "https://cdn.bancentral.gov.do/documents/estadisticas/precios/documents/ipc_base_2019-2020.xls"

inicio = time.time()
datos_ipc = ACQ.http_get(URL_IPC)
print(f"Descargados {len(datos_ipc):,} bytes en {time.time() - inicio:.1f} s")
print("SHA-256:", hashlib.sha256(datos_ipc).hexdigest())

### Prueba B · error 404: falla de inmediato, sin reintentos

Pide un archivo que no existe en el mismo servidor. Como es un 4xx, `http_get` no reintenta: el error sube enseguida.

In [0]:
inicio = time.time()
try:
    ACQ.http_get("https://cdn.bancentral.gov.do/documents/estadisticas/precios/documents/archivo_que_no_existe.xls")
except urllib.error.HTTPError as e:
    print(f"HTTPError {e.code} después de {time.time() - inicio:.1f} s → no hubo reintentos")
except ACQ.SourceUnavailable as e:
    print("El servidor no respondió como se esperaba:", e)

### Prueba C · servidor que no existe: reintenta y termina en `SourceUnavailable`

Se usa un dominio `.invalid`, que por norma nunca existe. Para no esperar 30 s, se pasa `attempts=2`: un fallo, 10 s de espera, otro fallo y la excepción propia.

In [0]:
inicio = time.time()
try:
    ACQ.http_get("https://fuente-caida.invalid/datos.xlsx", timeout=10, attempts=2)
except ACQ.SourceUnavailable as e:
    print(f"SourceUnavailable después de {time.time() - inicio:.1f} s (2 intentos y 10 s de espera)")
    print("Mensaje:", str(e)[:200])

## 4 · `ckan_resources`: pedir la URL actual al portal de datos abiertos

datos.gob.do usa **CKAN**, una plataforma con API. En vez de guardar una URL fija que puede cambiar, se le pregunta a la API cuál es la URL actual de cada formato. La usan `04_MICM_Fuel_Prices_Download` y `05_ProConsumidor_Prices_Download`.

In [0]:
ver(ACQ.ckan_resources)

- `(r.get("format") or "").strip().lower()`: programación defensiva. Si el formato viene vacío o `None` usa `""`; luego quita espacios y pasa a minúsculas para que `"CSV "` y `"csv"` sean iguales.
- `fmt not in found`: se queda con el primer recurso de cada formato.
- `missing` (list comprehension): si falta un formato que el notebook necesita, lanza `RuntimeError`. Es un error del proyecto, no de la fuente.
- Devuelve una **tupla**: el diccionario de URLs y la fecha de última modificación del dataset.

In [0]:
try:
    urls, modificado = ACQ.ckan_resources("precios-de-los-combustibles", ["xlsx", "csv"])
    print("Última modificación del dataset:", modificado)
    for formato, url in urls.items():
        print(f"  {formato}: {url}")
except ACQ.SourceUnavailable as e:
    print("datos.gob.do no respondió ahora mismo (así se vería en el Job):", str(e)[:200])

## 5 · `_sha256_file`: la huella del contenido

El guion bajo inicial (`_`) es una convención de Python: función de uso interno del módulo.

**SHA-256** produce una huella de 64 caracteres. Si cambia un solo byte, la huella cambia por completo. Dos archivos con la misma huella tienen, en la práctica, el mismo contenido.

In [0]:
ver(ACQ._sha256_file)

- `"rb"`: lectura binaria, porque son Excel y PDF.
- `1 << 20` = 1,048,576 bytes = 1 MB. Lee de a 1 MB para no cargar archivos grandes completos en memoria.
- `iter(funcion, b"")`: llama a la función hasta que devuelva `b""` (fin del archivo). Es una forma corta de `while True` con `break`.
- `h.update(chunk)`: acumula el hash pedazo por pedazo. El resultado es igual al de leer todo de una vez; la prueba lo comprueba.

In [0]:
print("hola →", hashlib.sha256(b"hola").hexdigest())
print("Hola →", hashlib.sha256(b"Hola").hexdigest(), " (solo cambió una mayúscula)")

carpeta_prueba = tempfile.mkdtemp(prefix="acq_demo_") + "/"
ruta = carpeta_prueba + "ipc_completo.xls"
with open(ruta, "wb") as fh:
    fh.write(datos_ipc)

por_pedazos = ACQ._sha256_file(ruta)
de_una_vez = hashlib.sha256(datos_ipc).hexdigest()
print("\nHash leyendo de a 1 MB:", por_pedazos)
print("Hash en memoria:       ", de_una_vez)
assert por_pedazos == de_una_vez
print("Son iguales")

## 6 · `_archived_files`: un generador que recorre el archivo histórico

In [0]:
ver(ACQ._archived_files)

- `os.walk` recorre todas las subcarpetas (año/mes).
- `dirs[:] = [...]` modifica la lista **en el mismo objeto**. Eso le dice a `os.walk` que no entre en carpetas que empiezan con `_`, como `_checkpoints` de Auto Loader. Con `dirs = [...]` (sin `[:]`) se crearía otra lista y `os.walk` entraría igual.
- `yield` convierte la función en **generador**: entrega una ruta a la vez en lugar de armar toda la lista en memoria.

In [0]:
os.makedirs(carpeta_prueba + "_checkpoints", exist_ok=True)
open(carpeta_prueba + "_checkpoints/offset.json", "w").write("{}")

print("Archivos que ve _archived_files (no entra en _checkpoints):")
for p in ACQ._archived_files(carpeta_prueba):
    print("  ", p[len(carpeta_prueba):])
print("Tipo de objeto que devuelve:", type(ACQ._archived_files(carpeta_prueba)).__name__)

## 7 · `land`: guardar solo lo nuevo

Es la función central. Recibe los bytes descargados, el nombre del archivo y la carpeta raíz del archivo histórico (*landing*).

In [0]:
ver(ACQ.land)

**Paso a paso**

1. `now = now or datetime.now(ZoneInfo("America/Santo_Domingo"))`: usa la hora de RD para elegir la carpeta del mes. El parámetro `now` es opcional para poder probar con una fecha fija (lo hacen las pruebas de abajo). No se pone `datetime.now()` directamente en la firma porque se evaluaría una sola vez, al importar el módulo.
2. Calcula el SHA-256 de los bytes nuevos.
3. Recorre el archivo histórico. Primero compara el **tamaño** (no cuesta nada) y solo si coincide calcula el hash del archivo guardado. El `and` se detiene en cuanto la primera condición es falsa (cortocircuito).
4. Si encuentra el mismo contenido → devuelve `SKIPPED` y no escribe nada.
5. `name.rsplit(".", 1)`: separa nombre y extensión por el último punto.
6. Si ya existe un archivo con **el mismo nombre pero otro contenido**, la fuente lo republicó (a veces con correcciones). El nuevo se guarda como `nombre__AAAA-MM-DD.ext`. El anterior **no se toca**.
7. Crea la carpeta `AAAA/MM/` (`exist_ok=True` evita error si ya existe).
8. Si incluso ese nombre ya existe el mismo día, agrega hora, minuto y segundo.
9. Escribe los bytes tal cual llegaron (`"wb"`). Bronze recibe el archivo crudo.
10. Devuelve un diccionario con el resultado. Los notebooks lo muestran al terminar, así cada corrida deja evidencia.

### Prueba: los cuatro casos

Se usa una carpeta temporal nueva y fechas fijas para que el resultado sea siempre el mismo.

In [0]:
landing = tempfile.mkdtemp(prefix="landing_demo_") + "/"
RD = ZoneInfo("America/Santo_Domingo")
original = b"periodo,valor\n2026-10,100\n"
corregido = b"periodo,valor\n2026-10,101\n"     # mismo nombre, un dato corregido

casos = [
    ("1. Primera descarga",                       original,  datetime(2026, 11, 2, 7, 10, tzinfo=RD)),
    ("2. El Job corre otra vez (mismo contenido)", original,  datetime(2026, 11, 2, 9, 0, tzinfo=RD)),
    ("3. La fuente republica con una corrección",  corregido, datetime(2026, 11, 9, 7, 10, tzinfo=RD)),
    ("4. Llega la corrección otra vez",            corregido, datetime(2026, 11, 16, 7, 10, tzinfo=RD)),
]
filas = []
for caso, contenido, fecha in casos:
    r = ACQ.land(contenido, "precios.csv", landing, now=fecha)
    filas.append((caso, r["status"], r.get("path") or r.get("reason")))

display(spark.createDataFrame(filas, "caso string, resultado string, detalle string"))

print("Archivo histórico final:")
for p in sorted(ACQ._archived_files(landing)):
    print("  ", p[len(landing):])

**Lo que debes ver:** el caso 1 descarga (`DOWNLOADED`), el 2 se salta (`SKIPPED`), el 3 guarda `precios__2026-11-09.csv` sin borrar el original y el 4 se salta porque esa versión ya existe. Al final hay **dos archivos**, uno por cada contenido distinto. Eso es idempotencia: correr una o varias veces deja el mismo resultado.

## 8 · `source_unavailable`: la caída como advertencia

Esta función **no se ejecuta aquí**, porque escribe en la tabla de calidad y termina el notebook con `dbutils.notebook.exit`. Se muestra el código y luego los registros reales que dejó en la plataforma.

In [0]:
ver(ACQ.source_unavailable)

- `str(error)[:1000]`: corta el mensaje para no llenar la tabla.
- Agrega una fila a `quality.source_availability` (tabla Delta, modo `append`).
- El `try/except Exception` es a propósito: si guardar el registro falla, solo se imprime. **Registrar una advertencia nunca debe convertirla en un fallo.** Es el único `except` amplio del módulo.
- `dbutils.jobs.taskValues.set("downloaded", False)`: los *task values* pasan información entre tareas de un Job. Las siguientes saben que no llegó nada nuevo.
- `dbutils.notebook.exit(...)`: termina el notebook con éxito y devuelve un JSON con el motivo. La fuente conserva sus últimos datos y el resto del pipeline sigue.

In [0]:
display(spark.sql("""
    SELECT source AS fuente, status AS estado, left(detail, 140) AS detalle, checked_ts AS revisado_utc
    FROM rd_economic_intelligence.quality.source_availability
    ORDER BY checked_ts DESC
    LIMIT 10
"""))

## 9 · Cómo lo usa un notebook de descarga

Todos los notebooks de `src/bronze/*_Download` siguen el mismo patrón. Este es el de `02_BCRD_CPI_Download`, resumido:

```python
try:
    results = {}
    for name in FILES:
        data = ACQ.http_get(BASE + name)                     # 1. descargar con reintentos
        if "Período" not in first_rows_text(name, data):     # 2. validar la estructura
            raise ValueError(f"{name}: structural contract failed")
        results[name] = ACQ.land(data, name, LANDING)        # 3. guardar solo lo nuevo
    downloaded = any(r["status"] == "DOWNLOADED" for r in results.values())
    dbutils.jobs.taskValues.set(key="downloaded", value=downloaded)
    dbutils.notebook.exit(json.dumps({"status": "DOWNLOADED" if downloaded else "SKIPPED", "files": results}))
except ACQ.SourceUnavailable as e:                           # 4. solo la caída de la fuente es advertencia
    ACQ.source_unavailable(dbutils, spark, "bcrd_cpi", e)
```

| Notebook | Funciones que usa |
|---|---|
| `01_BCRD_Exchange_Rates_Download` | `http_get`, `land`, `source_unavailable` |
| `02_BCRD_CPI_Download` | `http_get`, `land`, `source_unavailable` |
| `04_MICM_Fuel_Prices_Download` | `ckan_resources`, `http_get`, `land`, `source_unavailable` |
| `05_ProConsumidor_Prices_Download` | `ckan_resources`, `http_get`, `land`, `source_unavailable` |
| `06_MEM_Electricity_Download` | `http_get`, `land`, `source_unavailable` |
| `07_MA_Agricultural_Prices_Download` | `http_get` (incluida la API CKAN), `land`, `source_unavailable` |

**Después de `land`:** Bronze usa Auto Loader (`src/shared/ingestion.py`) con un *checkpoint* para registrar cada archivo nuevo una sola vez. Es la segunda capa contra duplicados.

## Limpieza

In [0]:
import shutil
for carpeta in (carpeta_prueba, landing):
    shutil.rmtree(carpeta, ignore_errors=True)
print("Carpetas temporales borradas. No se escribió nada en el Volume ni en las tablas.")